In [2]:
import pandas as pd
import numpy as np

In [12]:
def calc_joint(df_1, a, df_2, b):
    tab = pd.crosstab(df_1[a], df_2[b])
    Pxy = tab / tab.values.sum()
    return Pxy, Pxy.sum(1), Pxy.sum(0)

In [13]:
# def mi_discrete(Pxy, Px, Py, base=2.0):
#     P     = Pxy.values
#     denom = Px.values[:, None] * Py.values[None, :]
#     nz    = P > 0
#     # Add a small epsilon to prevent log(0) from invalid divisions
#     denom[denom == 0] = 1e-15
#     mi    = (P[nz] * (np.log(P[nz]) - np.log(denom[nz]))).sum()
#     return mi / np.log(base)

def mi_discrete(Pxy, Px, Py, base=2.0):
    P     = Pxy.values
    # Align Px and Py to Pxy's row/column order
    Px = Px.reindex(Pxy.index, fill_value=0).values
    Py = Py.reindex(Pxy.columns, fill_value=0).values
    denom = Px[:, None] * Py[None, :]
    denom[denom == 0] = 1e-15
    nz = P > 0
    mi = (P[nz] * (np.log(P[nz]) - np.log(denom[nz]))).sum()
    return mi / np.log(base)


In [14]:
def calculate_I(df_1, df_2, gene_1, gene_2):
    Pxy, Px, Py = calc_joint(df_1, gene_1, df_2, gene_2)
    mi = mi_discrete(Pxy, Px, Py, base=2.0)
    return mi

In [6]:
# def specific_info(df, X, Z, base=2.0):
#     N      = len(df)
#     joint  = df.groupby([Z, X]).size().unstack(fill_value=0).astype(float) #calculates the frequency table of number of occurences of the combination
#     p_xz   = joint / N
#     p_z    = p_xz.sum(1) #calculate sum across columns
#     p_x    = p_xz.sum(0) #calculate sum across rows
#     p_xg_z = p_xz.div(p_z, axis=0).fillna(0.0)
#     ratio  = p_xg_z.div(p_x, axis=1).replace([np.inf, -np.inf], 0.0).fillna(0.0)
#     lr     = np.log(ratio.clip(lower=1e-15)) / np.log(base) # clip to avoid log(0)
#     return (p_xg_z * lr).sum(1)

def specific_info(df_1, df_2, x, z, base=2.0):
    N = len(df_1)

    # joint[z, x]
    joint = df.groupby([z, x]).size().unstack(fill_value=0).astype(float)
    p_zx = joint / N

    # p(z), p(x)
    p_z = p_zx.sum(axis=1)  # axis=1 => sum over x
    p_x = p_zx.sum(axis=0)  # axis=0 => sum over z

    # p(x|z)
    p_x_given_z = p_zx.div(p_z, axis=0).dropna()

    # log(p(x|z)/p(x))
    ratio = p_x_given_z.div(p_x, axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    log_term = np.log(ratio.clip(lower=1e-15)) / np.log(base)

    return (p_x_given_z * log_term).sum(axis=1)  # output indexed by z



In [7]:
# def calculate_R(df, gene_i, gene_j, gene_k, base = 2.0):
#     I1 = specific_info(df, gene_i, gene_j)
#     I2 = specific_info(df, gene_k, gene_j)
#     I1, I2 = I1.align(I2, fill_value=0)
#     p_gene_j = df[gene_j].value_counts(normalize=True).reindex(I1.index, fill_value=0)
#     r = (p_gene_j * np.minimum(I1, I2)).sum()
#     # print(f"p_gene_k = {p_gene_j}, I1 = {I1}, I2 = {I2} and r = {r}")
#     return r

def calculate_R(df_1, gene_i, gene_j, gene_k, base = 2.0):
    I1 = specific_info(df, gene_i, gene_j)
    I2 = specific_info(df, gene_k, gene_j)
    I1, I2 = I1.align(I2, fill_value=0)
    p_gene_j = df[gene_j].value_counts(normalize=True).reindex(I1.index, fill_value=0)

    # print(f"\n--- Redundancy for {gene_i}, {gene_j} given {gene_k} ---")
    # print("p(z):", p_gene_j.values)
    # print("I_spec(z; X):", I1.values)
    # print("I_spec(z; Y):", I2.values)
    # print("min(I1, I2):", np.minimum(I1, I2).values)
    # print("weighted min:", (p_gene_j * np.minimum(I1, I2)).values)

    r = (p_gene_j * np.minimum(I1, I2)).sum()
    return r


In [8]:
path_to_sim="/home/mzo5929/Keerthana/grnInference/simulation_data/median_parameter_simulations/new_simulation/df_rows_0_1_2_31072025_150153_ncells_10000_Fan_out_3626552d.csv"
df = pd.read_csv(path_to_sim)

In [9]:
t1 = 5
t2 = 20

# mRNA columns
mRNA_cols = [col for col in df.columns if col.endswith("_mRNA")]

# t1 
df_t1 = df[df["time_step"] == t1]
rep_0_t1 = df_t1[df_t1["replicate"] == 1].reset_index(drop=True)[mRNA_cols]
rep_1_t1 = df_t1[df_t1["replicate"] == 2].reset_index(drop=True)[mRNA_cols]
expr_t1_all = pd.concat([rep_0_t1, rep_1_t1]).reset_index(drop=True)

# t2
df_t2 = df[df["time_step"] == t2]
rep_0_t2 = df_t2[df_t2["replicate"] == 1].reset_index(drop=True)[mRNA_cols]
rep_1_t2 = df_t2[df_t2["replicate"] == 2].reset_index(drop=True)[mRNA_cols]
expr_t2_all = pd.concat([rep_0_t2, rep_1_t2]).reset_index(drop=True)

In [7]:
# Get only mRNA columns
mRNA_cols = [col for col in df.columns if col.endswith("_mRNA")]
df_t1_expr = df_t1[mRNA_cols]
df_t1_expr.to_csv("/home/mzo5929/Keerthana/grnInference/code/InformationMeasures.jl/keerthana_test/expr_t1_input.csv", index=False)

In [79]:
import numpy as np
from itertools import permutations
import pandas as pd

from joblib import Parallel, delayed
import numpy as np

def calculate_shuffled_mi(df, gene_x, gene_y, base=2.0, seed=None):
    """Compute MI for one shuffled instance of gene_x."""
    rng = np.random.default_rng(seed)
    shuffled = df[gene_x].values.copy()
    rng.shuffle(shuffled)
    shuffled_df = df.copy()
    shuffled_df[gene_x] = shuffled
    return calculate_I(shuffled_df, gene_x, gene_y)

def calculate_mi_pvalue_parallel(df, gene_x, gene_y, n_iter=10000, base=2.0, n_jobs=10, seed=101010):
    """Compute true MI and null distribution in parallel."""
    true_mi = calculate_I(df, gene_x, gene_y)

    # Generate independent seeds for each shuffle
    seed_seq = np.random.SeedSequence(seed)
    seeds = seed_seq.spawn(n_iter)
    int_seeds = [s_entropy.generate_state(1)[0] for s_entropy in seeds]

    shuffled_mis = Parallel(n_jobs=n_jobs)(
        delayed(calculate_shuffled_mi)(df, gene_x, gene_y, base, s)
        for s in int_seeds
    )

    shuffled_mis = np.array(shuffled_mis)
    pval = np.mean(shuffled_mis >= true_mi)
    return true_mi, pval, shuffled_mis.mean(), shuffled_mis.std()

# --- Matrix setup ---
n_genes = len(expr_t1_all.columns)
gene_list = list(expr_t1_all.columns)

I_matrix = np.zeros((n_genes, n_genes))
pval_matrix = np.ones((n_genes, n_genes))  # initialize p-values to 1
R_matrix = np.zeros((n_genes, n_genes))
U_matrix = np.zeros((n_genes, n_genes))

# --- Loop through all triplets ---
list_of_tuples = list(permutations(range(n_genes), 3))

from tqdm.auto import tqdm
for i, j, k in tqdm(list_of_tuples):
    gene_i, gene_j, gene_k = gene_list[i], gene_list[j], gene_list[k]

    # Compute real MI and p-value via 10000 shuffles
    mi, pval, null_mean, null_std = calculate_mi_pvalue_parallel(df_t1, gene_i, gene_j, n_iter=1000)
    I_matrix[i, j] = mi
    pval_matrix[i, j] = pval  # optional: store p-value

    # Redundancy
    R_matrix[j, i] = calculate_R(df_t1, gene_list[i], gene_list[j], gene_list[k])  # Use gene names for calculation

# Unique information
U_matrix = I_matrix - R_matrix


  0%|          | 0/6 [00:00<?, ?it/s]

In [80]:
I_matrix

array([[0.        , 0.00530802, 0.00419916],
       [0.00530802, 0.        , 0.01604317],
       [0.00419916, 0.01604317, 0.        ]])

In [86]:
R_matrix

array([[0.        , 0.00350823, 0.00350823],
       [0.00528041, 0.        , 0.00528041],
       [0.00417203, 0.00417203, 0.        ]])

In [81]:
pval_matrix

array([[1.   , 0.   , 0.013],
       [0.   , 1.   , 0.   ],
       [0.006, 0.   , 1.   ]])

In [82]:
U_matrix

array([[0.00000000e+00, 1.79979027e-03, 6.90934838e-04],
       [2.76051401e-05, 0.00000000e+00, 1.07627581e-02],
       [2.71287595e-05, 1.18711371e-02, 0.00000000e+00]])

In [87]:
#i = A, j = B, k = C
i = "gene_1_mRNA"
j = "gene_2_mRNA"
k = "gene_3_mRNA"

I_A_B = calculate_I(df_t1, i, j)
R_B_A_C = calculate_R(df_t1, i, j, k)
U_B_A = I_A_B - R_B_A_C
U_B_A/I_A_B

0.005200650993508072

In [88]:
#i = A, j = B, k = C
i = "gene_1_mRNA"
j = "gene_2_mRNA"
k = "gene_3_mRNA"

I_B_A = calculate_I(df_t1, j, i)
R_A_B_C = calculate_R(df_t1, j, i, k)
U_A_B = I_B_A - R_A_B_C
U_A_B/I_B_A

0.3390702234669593

In [89]:
i = "gene_1_mRNA"
j = "gene_2_mRNA"
k = "gene_3_mRNA"

I_C_A = calculate_I(df_t1, k, i)
R_A_C_B = calculate_R(df_t1, k, i, j)
U_A_C = I_A_C - R_A_C_B
U_A_C/I_C_A

-0.24833749841563213

In [90]:
i = "gene_1_mRNA"
j = "gene_2_mRNA"
k = "gene_3_mRNA"

I_A_C = calculate_I(df_t1, i, k)
R_C_A_B = calculate_R(df_t1, i, k, j)
U_C_A = I_A_C - R_C_A_B
U_C_A/I_A_C

0.006460519257382508

In [91]:
#i = A, j = B, k = C
i = "gene_1_mRNA"
j = "gene_2_mRNA"
k = "gene_3_mRNA"

I_C_B = calculate_I(df_t1, k, j)
R_B_C_A = calculate_R(df_t1, k, j, i)
U_B_C = I_C_B - R_B_C_A
U_B_C/I_C_B

0.6708623427920083

In [92]:
#i = A, j = B, k = C
i = "gene_1_mRNA"
j = "gene_2_mRNA"
k = "gene_3_mRNA"

I_B_C = calculate_I(df_t1, j, k)
R_C_B_A = calculate_R(df_t1, j, k, i)
U_C_B = I_B_C - R_C_B_A
U_C_B/I_B_C

0.739949630334455

In [37]:
U_B_A/I_A_B

0.3553086053499117

In [23]:
I_matrix

array([[0.        , 0.00826091, 0.00246542],
       [0.00826091, 0.        , 0.00675255],
       [0.00246542, 0.00675255, 0.        ]])

In [24]:
R_matrix

array([[0.        , 0.00532574, 0.00246542],
       [0.00246542, 0.        , 0.00246542],
       [0.00246542, 0.00532574, 0.        ]])

In [25]:
U_matrix

array([[0.00000000e+00, 2.93517327e-03, 1.14491749e-16],
       [5.79549551e-03, 0.00000000e+00, 4.28712887e-03],
       [8.10983225e-17, 1.42680663e-03, 0.00000000e+00]])

In [83]:
result_matrix = np.divide(U_matrix, I_matrix, out=np.zeros_like(I_matrix), where=I_matrix != 0)
result_matrix

array([[0.        , 0.33907022, 0.16454117],
       [0.00520065, 0.        , 0.67086234],
       [0.00646052, 0.73994963, 0.        ]])

# code from Chan et al
